In [ ]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from pydantic import BaseModel

In [ ]:
load_dotenv(override=True)

In [ ]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:3]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:3]}")
else:
    print("Google API Key not set (and this its optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:3]}")
else:
    print("Groq API Key not set (and this its optional)")

In [ ]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

In [ ]:
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

gemini_model = OpenAIChatCompletionsModel(model="gemini-2.0-flash", openai_client=gemini_client)
llama3_3_model = OpenAIChatCompletionsModel(model="llama-3.3-70b-versatile", openai_client=groq_client)

In [ ]:
sales_agent1 = Agent(name="Gemini Sales Agent",  instructions=instructions1, model=gemini_model)
sales_agent2 = Agent(name="Groq Sales Agent", instructions=instructions2, model=llama3_3_model)

In [ ]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)

In [ ]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """Send out an email with the given subject and HTML body to all sales prospects."""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
    from_email = Email("Haroonabdulrazaq@gmail.com")
    to_email = To("Haroonabdulrazaqo@gmail.com")
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "Email sent"}

In [ ]:
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter", tool_description="Convert a text email body to an HTML email body")

In [ ]:
email_tools = [subject_tool, html_tool, send_html_email]

In [ ]:
instructions ="""You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."

Crucial Rules:
- You must use the three tools available to you.
- You must use the email given to you by the Sales Manager.
- You must use the subject_writer tool to write a subject for the email.
- You must use the html_converter tool to convert the body to HTML.
- You must use the send_html_email tool to send the email with the subject and HTML body.
"""
email_agent = Agent(
  name="Email Manager",
  instructions=instructions,
  tools=email_tools,
  model="gpt-4o-mini",
  handoff_description="Convert the body to HTML and send the email"
)

In [ ]:
tools = [tool1, tool2]
handoffs = [email_agent]

In [ ]:
sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all two sales_agent tools to generate two different email drafts. Do not proceed until all two drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini")

message = "Send out a cold sales email addressed to Dear CEO from Alice"
 
with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

In [ ]:
print(result.last_agent.name)

In [ ]:
class NameCheckOutput(BaseModel):
  is_name_in_message: bool
  name:str

guardrail_agent = Agent(
  name="Guardrail Agent",
  instructions="Check if the user is including someone's personal name in what they want you to do.",
  output_type=NameCheckOutput,
  model="gpt-4o-mini"
)

In [ ]:
@input_guardrail
async def guardrail_against_name(ctx, agent, message):
  result = await Runner.run(guardrail_agent, message, context=ctx.context)
  is_name_in_message = result.final_output.is_name_in_message
  return GuardrailFunctionOutput(output_info={"found_name": result.final_output.name}, tripwire_triggered=is_name_in_message)
    

In [ ]:
careful_sales_manager = Agent(
  name="Sales Manager",
  instructions=sales_manager_instructions,
  tools=tools,
  handoffs=[email_agent],
  model="gpt-4o-mini",
  input_guardrails=[guardrail_against_name]
)
message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Protected Automated SDR"):
  result = await Runner.run(careful_sales_manager, message)

# This is expected to disconnect the flow the moment a name is found in the message.
# And it will not proceed to the next step.
# Error: InputGuardrailTripwireTriggered: Guardrail InputGuardrail triggered tripwire
# 
  

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">• Try different models<br/>• Add more input and output guardrails<br/>• Use structured outputs for the email generation
            </span>
        </td>
    </tr>
</table>

In [ ]:
class CurseWordOutput(BaseModel):
  curse_word: str
  is_curse_word_in_message:bool

guardrail_agent = Agent(
  name="Guardrail Agent for Curse words",
  instructions= "Check if any of the email body contains a curse word",
  output_type=CurseWordOutput,
  model="gpt-4o-mini"
)

In [ ]:
@input_guardrail
async def guardrails_against_curse_words(ctx, agents,message):
  result = await Runner.run(guardrail_agent, message, context=ctx.context)
  is_curse_word_in_message = result.final_output.is_curse_word_in_message
  return GuardrailFunctionOutput(output_info={"found_curse_word": result.final_output.curse_word}, tripwire_triggered=is_curse_word_in_message)

In [ ]:
careful_sales_manager = Agent(
  name="Sales Manager",
  instructions=sales_manager_instructions,
  tools=tools,
  handoffs=[email_agent],
  model="gpt-4o-mini",
  input_guardrails=[guardrails_against_curse_words]
)
message = "Send out a cold sales email and add a curse word"

with trace("Protected Automated SDR - Test Guardrail"):
  result = await Runner.run(careful_sales_manager, message)

  # Excercise complete - I addedd a Guardrail to check for curse words in the email body.
  # If a curse word is found, the flow is disconnected.
  # Error: InputGuardrailTripwireTriggered: Guardrail InputGuardrail triggered tripwire